# NER job: verification tests

Run top to bottom on a GPU instance (ml.g6 / L4), from the `notebooks/` folder, after a kernel restart.
Each check prints `PASS` / `FAIL`; the last cell prints a summary.

| Section | What it checks | GPU | Writes |
|---|---|---|---|
| 1 | Config, secrets, Redshift read | no | nothing |
| 2 | Messages and candidate matching; 2b: ticker noise filter | no | nothing |
| 3 | Tokenizer: single `<bos>`, JSON answer length vs old limit, prompt lengths, batch plan | no | nothing |
| 4 | Old (notebook) vs new full-format vs new compact-format on a sample | yes, 3 model loads in child processes | `ner_test_utils.py` next to this notebook |
| 5 | Full production run + output shape | yes, 1 model load in a child process | nothing |
| 6 | COPY into a scratch table (created and dropped here) | no | scratch table only |

The model is never loaded in this kernel, so GPU memory is always released between sections.
Production tables are never touched.

## 0. Setup

In [ ]:
import logging
import os
import sys
import time
from pathlib import Path

# Absolute paths, so spawned child processes (sections 4-5) see them too
sys.path.insert(0, str(Path("../src").resolve()))
sys.path.insert(0, os.getcwd())
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s - %(message)s", force=True)

import pandas as pd
import torch
from transformers import AutoTokenizer

from asksa_entity import EntityValidatorLLM, build_prompts, build_validated_df
from asksa_entity.llm_validate import ANSWER_PREFIXES
from asksa_jobs import ner_job as nj
from asksa_jobs.common import (
    export_hf_token,
    fetch_df,
    load_df_to_redshift,
    load_settings,
    read_query,
    redshift_connect,
)

pd.set_option("display.max_colwidth", 200)
RESULTS = []


def check(name, ok, details=""):
    """Record and print one test result."""
    status = "PASS" if ok else "FAIL"
    RESULTS.append((status, name, details))
    print(f"{status} | {name}" + (f" | {details}" if details else ""))


def check_gpu_free(before_what, min_free_gb=20):
    """The 9B model needs ~18 GB; less free memory means CPU offload or OOM."""
    free_gb = torch.cuda.mem_get_info()[0] / 1024**3
    check(f"GPU free before {before_what}", free_gb >= min_free_gb, f"{free_gb:.1f} GB free")
    if free_gb < min_free_gb:
        raise RuntimeError("GPU memory is in use: restart the kernel and check !nvidia-smi")


check("CUDA available", torch.cuda.is_available())
GPU_TOTAL_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {torch.cuda.get_device_name(0)}, {GPU_TOTAL_GB:.1f} GB")

## 1. Config, secrets, Redshift read

In [ ]:
settings = load_settings()
params = nj.NerParams()
print(params)

token = export_hf_token(settings)
check("HF token loaded from Secrets Manager", token.startswith("hf_"), f"{token[:5]}...")

tickers = fetch_df(settings, read_query(settings, nj.TICKERS_QUERY_FILE))
check("tickers query works", len(tickers) > 0, f"{len(tickers)} rows")

## 2. Messages and candidate matching (no GPU)

In [ ]:
messages = nj.load_messages(settings)
check("there are new messages to test on", len(messages) > 0, f"{len(messages)} messages")
check("one row per message", not messages.duplicated(subset=nj.MESSAGE_KEY).any())

groups = nj.build_candidates(settings, messages, params)
check("some messages have candidates", len(groups) > 0, f"{len(groups)} of {len(messages)}")

n_cands = groups["payload"].map(len)
print(n_cands.describe(percentiles=[0.5, 0.9, 0.99]).round(1).to_string())
groups[["msg", "payload"]].head()

## 2b. Ticker noise filter (no GPU)

Ticker matches dropped before the LLM because they are ordinary words with no ticker signal
(see `src/asksa_entity/ticker_noise.py`). Review the samples: **is any of them a ticker the user meant?**
If a word shows up there wrongly, remove it from `COMMON_WORDS`.

In [ ]:
from dataclasses import replace

from asksa_entity import explain_ticker_noise

groups_off = nj.build_candidates(settings, messages, replace(params, ticker_noise_filter=False))
cands_on, cands_off = groups["payload"].map(len).sum(), groups_off["payload"].map(len).sum()
print(f"messages with candidates: {len(groups_off)} -> {len(groups)} (LLM prompts saved: {len(groups_off) - len(groups)})")
print(f"candidates: {cands_off} -> {cands_on} ({cands_off - cands_on} dropped)")

tickers_all = fetch_df(settings, read_query(settings, nj.TICKERS_QUERY_FILE))
from asksa_entity import ExtractConfig, build_resources, get_topics_df

R = build_resources(
    tickers_all[["ticker_id", "company_name"]],
    tickers_all[["sector_name", "industry_name"]].drop_duplicates(),
    get_topics_df(),
    ExtractConfig(batch_size=params.extract_batch_size, max_ngram_fallback=params.max_ngram_fallback),
)
noise = explain_ticker_noise(messages, R)
print(noise["rule"].value_counts().to_string())
print()
print("most dropped words:", noise["token"].str.lower().value_counts().head(30).to_dict())
check("noise filter drops something", len(noise) > 0, f"{len(noise)} matches")
noise.sample(n=min(40, len(noise)), random_state=1)[["rule", "token", "ticker", "msg"]]

## 3. Tokenizer checks (no model)

Uses the production prompt code with the tokenizer only.
3.2 shows why the old `max_new_tokens=30` loses data: it is the token length of a correct JSON answer.

In [ ]:
def tokenizer_only(params, token, use_chat_template=True):
    """EntityValidatorLLM without the model: same rendering/tokenization/batching code."""
    v = EntityValidatorLLM.__new__(EntityValidatorLLM)
    v.use_chat_template = use_chat_template
    v.output_format = params.output_format
    v.answer_prefix = ANSWER_PREFIXES[params.output_format] if params.prefill_answer else ""
    v.batch_size = params.batch_size
    v.max_batch_tokens = params.max_batch_tokens
    v.max_input_tokens = params.max_input_tokens
    v.tokenizer = AutoTokenizer.from_pretrained(params.model_id, token=token)
    return v


tok_v = tokenizer_only(params, token)
tok = tok_v.tokenizer

# 3.1 single <bos>
ids = tok_v._tokenize(tok_v._render(["hi"]))["input_ids"][0]
first = tok.convert_ids_to_tokens(ids[:3])
check("prompt starts with exactly one <bos>", first[0] == "<bos>" and first[1] != "<bos>", str(first))

# 3.2 token length of a correct answer with 1..3 entities
example_ids = ["AAPL", "TSLA", "NVDA"]
for k in [1, 2, 3]:
    ents = ", ".join(f'{{"idx": {i}, "entity_type": "ticker", "entity_id": "{e}"}}' for i, e in enumerate(example_ids[:k]))
    answer = f'{{"entities": [{ents}]}}'
    n = len(tok(answer, add_special_tokens=False)["input_ids"])
    compact = "{" + '"keep":[' + ",".join(str(i) for i in range(k)) + "]}"
    n_compact = len(tok(compact, add_special_tokens=False)["input_ids"])
    print(f"{k} entities: full {n} tokens -> {'fits' if n <= 30 else 'CUT'} with old max_new_tokens=30; compact {n_compact} tokens")

# 3.3 prompt lengths
prompt_len = pd.Series(tok_v.prompt_lengths(groups["prompt"].tolist()))
print(prompt_len.describe(percentiles=[0.5, 0.9, 0.99]).round(0).to_string())
over = int((prompt_len > params.max_input_tokens).sum())
check(f"prompts over max_input_tokens={params.max_input_tokens} (skipped) below 1%",
      over <= 0.01 * len(prompt_len), f"{over} of {len(prompt_len)}")

# 3.4 batch plan
eligible = {i: n for i, n in enumerate(prompt_len) if n <= params.max_input_tokens}
plan = tok_v._make_batches(eligible)
check("every batch within token budget",
      all(len(b) == 1 or len(b) * max(eligible[i] for i in b) <= params.max_batch_tokens for b in plan))
check("batch plan covers every eligible row once", sorted(i for b in plan for i in b) == sorted(eligible))
print(f"generate() calls: {len(plan)} vs {len(eligible)} with batch_size=1 in the notebook")

## 4. Old vs new LLM settings on a sample

Same candidates and validation rules for all three variants:
- **old** (notebook): raw prompt without chat template, fp16, `max_new_tokens=30`, `repetition_penalty=1.05`, one message per call.
- **full**: chat template, bf16, original answer format `{"entities":[{idx, entity_type, entity_id}]}`.
- **new** (production `NerParams`): like full, but compact answer format `{"keep":[0,"3:BRK.B"]}`.

full/new: each row stops when its JSON is complete or it starts repeating,
cut answers keep their complete items, token-budget batches.
Each variant runs in its own child process (the next cell writes the helper module).
The key comparison is **full vs new**: same model behaviour, only the answer format differs.

In [ ]:
%%writefile ner_test_utils.py
"""Helpers for ner_tests.ipynb: run LLM variants in child processes."""

import dataclasses
import multiprocessing as mp
import time
from concurrent.futures import ProcessPoolExecutor

LEGACY = dict(
    use_chat_template=False,
    output_format="full",
    prefill_answer=False,
    max_new_tokens=30,
    new_tokens_per_candidate=None,
    salvage_truncated_json=False,
    stop_on_complete_json=False,
    repetition_penalty=1.05,
    batch_size=1,
    max_batch_tokens=None,
    max_input_tokens=None,
)


def validate_variant(variant, messages, payloads, params, token):
    """Build prompts for the variant's format, load it, run all, report time and memory.

    old: the original notebook; full: new settings with the original answer
    format; new: production NerParams.
    """
    import torch

    from asksa_entity import EntityValidatorLLM, build_prompts
    from asksa_jobs.ner_job import make_validator

    start = time.time()
    if variant == "old":
        validator = EntityValidatorLLM(model_id=params.model_id, token=token, dtype=torch.float16, **LEGACY)
    elif variant == "full":
        full = dataclasses.replace(params, output_format="full", new_tokens_base=32, new_tokens_per_candidate=40)
        validator = make_validator(full, token)
    elif variant == "new":
        validator = make_validator(params, token)
    else:
        raise ValueError(f"Unknown variant: {variant}")
    load_time = time.time() - start

    prompts = build_prompts(messages, payloads, validator.output_format)
    torch.cuda.reset_peak_memory_stats()
    start = time.time()
    outputs = validator.run(prompts, n_candidates=[len(p) for p in payloads])
    return {
        "outputs": outputs,
        "load_time": load_time,
        "run_time": time.time() - start,
        "peak_gb": torch.cuda.max_memory_allocated() / 1024**3,
    }


def run_in_subprocess(func, *args):
    """Run func(*args) in a fresh spawned process; its errors are re-raised here."""
    ctx = mp.get_context("spawn")
    with ProcessPoolExecutor(max_workers=1, mp_context=ctx) as pool:
        return pool.submit(func, *args).result()

In [ ]:
import importlib

importlib.invalidate_caches()  # pick up the file written by the previous cell
from ner_test_utils import run_in_subprocess, validate_variant

SAMPLE_SIZE = 50
N_EXTREME = 6
SEED = 42

g = groups.reset_index(drop=True)
longest = prompt_len.sort_values(ascending=False).index[:N_EXTREME]
most_cands = n_cands.reset_index(drop=True).drop(index=longest).sort_values(ascending=False).index[:N_EXTREME]
picked = longest.union(most_cands)
rest = g.drop(index=picked)
sample = pd.concat(
    [g.loc[picked], rest.sample(n=min(SAMPLE_SIZE - len(picked), len(rest)), random_state=SEED)]
).reset_index(drop=True)
print(f"Sample: {len(sample)} of {len(g)} messages with candidates")

check_gpu_free("section 4")
VARIANTS = ["old", "full", "new"]
runs = {}
for variant in VARIANTS:
    runs[variant] = run_in_subprocess(
        validate_variant, variant, sample["msg"].tolist(), sample["payload"].tolist(), params, token
    )
    r = runs[variant]
    print(f"{variant}: load {r['load_time']:.0f}s, run {r['run_time']:.0f}s, peak {r['peak_gb']:.1f} GB")

In [ ]:
def entity_sets(groups_df, outputs):
    """Accepted (type, id) set per message, in groups_df order."""
    v = build_validated_df(groups_df, outputs)
    by_trace = {t: frozenset(zip(d["entity_type"], d["entity_id"])) for t, d in v.groupby("trace_id")}
    return groups_df["trace_id"].map(lambda t: by_trace.get(t, frozenset()))


def stats(outputs):
    n = len(outputs)
    return {
        "parse failed": sum(not o["parse_ok"] and not o["skipped"] for o in outputs),
        "cut (limit or looping)": sum(not o["finished"] and not o["skipped"] for o in outputs),
        "recovered from cut JSON": sum(o.get("salvaged", False) for o in outputs),
        "lost (failed, not recovered)": sum(
            not o["parse_ok"] and not o.get("salvaged", False) and not o["skipped"] for o in outputs
        ),
        "skipped (too long)": sum(o["skipped"] for o in outputs),
        "messages with >=1 entity": sum(len(o["entities"]) > 0 for o in outputs),
        "messages": n,
    }


outs = {v: runs[v]["outputs"] for v in VARIANTS}
sets = {v: entity_sets(sample, outs[v]).map(sorted) for v in VARIANTS}
old_ok = pd.Series([o["parse_ok"] for o in outs["old"]])

summary = pd.DataFrame({v: stats(outs[v]) for v in VARIANTS})
summary.loc["entities total"] = [sets[v].map(len).sum() for v in VARIANTS]
summary.loc["run time, s"] = [round(runs[v]["run_time"]) for v in VARIANTS]
summary.loc["peak GPU, GB"] = [round(runs[v]["peak_gb"], 1) for v in VARIANTS]
print(summary.to_string())
print()
for a, b in [("full", "new"), ("old", "new")]:
    same = sets[a] == sets[b]
    line = f"same entity set {a} vs {b}: {same.mean():.0%}"
    if a == "old":
        line += f" (where old JSON parsed: {same[old_ok].mean():.0%} of {old_ok.sum()} msgs)"
    print(line)
for v in ["full", "new"]:
    print(f"{v} stop reasons:", pd.Series([o["stop_reason"] for o in outs[v]]).value_counts().to_dict())

st = {v: stats(outs[v]) for v in VARIANTS}
lost = "lost (failed, not recovered)"
check("new: lost answers not worse than full", st["new"][lost] <= st["full"][lost], f"{st['full'][lost]} -> {st['new'][lost]}")
check("new: lost answers not worse than old", st["new"][lost] <= st["old"][lost], f"{st['old'][lost]} -> {st['new'][lost]}")
check("new: faster than full", runs["new"]["run_time"] < runs["full"]["run_time"],
      f"{runs['full']['run_time']:.0f}s -> {runs['new']['run_time']:.0f}s")
check("new: peak GPU memory below 95% of the card", runs["new"]["peak_gb"] < 0.95 * GPU_TOTAL_GB,
      f"{runs['new']['peak_gb']:.1f} of {GPU_TOTAL_GB:.1f} GB")

In [ ]:
# Answers of the production format that were cut: "repeating" = looping, "limit" = ran out of tokens
cut_answers = pd.DataFrame([
    {"stop_reason": o["stop_reason"], "n_candidates": len(sample["payload"].iloc[i]), "limit": o["max_new_tokens"],
     "entities": len(o["entities"]), "recovered": o.get("salvaged", False), "answer_end": o["text"][-300:]}
    for i, o in enumerate(outs["new"]) if not o["finished"] and not o["skipped"]
])
display(cut_answers)

# Review by hand: full vs new answer format (the decision), which set fits the message better?
review = pd.DataFrame({
    "msg": sample["msg"].astype(str).str.slice(0, 200),
    "n_candidates": sample["payload"].map(len),
    "old": sets["old"],
    "full": sets["full"],
    "new": sets["new"],
})
disagreements = review[sets["full"] != sets["new"]]
disagreements

## 5. Full production run (all messages, no Redshift write)

Same settings as the job, in a child process. Then the job's `prepare_output` here.

In [ ]:
check_gpu_free("section 5")
full = run_in_subprocess(validate_variant, "new", g["msg"].tolist(), g["payload"].tolist(), params, token)
outputs = full["outputs"]
nj.log_llm_stats(outputs)
validated = build_validated_df(g, outputs)
output = nj.prepare_output(validated)

full_stats = stats(outputs)
print(f"messages: {len(g)}, entities: {len(output)}, run {full['run_time'] / 60:.1f} min "
      f"(+ load {full['load_time']:.0f}s), peak {full['peak_gb']:.1f} GB")
print(full_stats)
check("output columns match the table", list(output.columns) == nj.OUTPUT_COLUMNS)
check("entity types are known", output["entity_type"].isin({"ticker", "sector", "industry", "topic"}).all(),
      str(sorted(output["entity_type"].unique())))
check("entity_id fits varchar(100)", (output["entity_id"].str.len() <= nj.ENTITY_ID_MAX_CHARS).all())
check("user_id / chat_id are integers", all(pd.api.types.is_integer_dtype(output[c]) for c in ["user_id", "chat_id"]))
check("msg_date filled", output["msg_date"].notna().all())
check("no duplicate (message, entity) rows",
      not output.duplicated(subset=["trace_id", "entity_type", "entity_id"]).any())
lost = full_stats["lost (failed, not recovered)"]
check("lost answers below 1%", lost <= 0.01 * len(g), f"{lost} of {len(g)}")
check("skipped prompts below 0.1%", full_stats["skipped (too long)"] <= 0.001 * len(g), str(full_stats["skipped (too long)"]))
check("peak GPU memory below 95% of the card", full["peak_gb"] < 0.95 * GPU_TOTAL_GB, f"{full['peak_gb']:.1f} GB")
output.head(10)

## 6. COPY into a scratch table

Creates `ds.tmp_ask_sa_ner_load_test` with the structure of the production table,
loads `output` through the production `load_df_to_redshift`, reads it back and drops the table.
Needs CREATE rights in the `ds` schema.

In [ ]:
SCRATCH_TABLE = "ds.tmp_ask_sa_ner_load_test"


def run_sql(sql):
    with redshift_connect(settings) as conn:
        cur = conn.cursor()
        cur.execute(sql)
        conn.commit()


run_sql(f"DROP TABLE IF EXISTS {SCRATCH_TABLE}")
run_sql(f"CREATE TABLE {SCRATCH_TABLE} (LIKE {settings.ner_table})")
try:
    loaded = load_df_to_redshift(settings, output, SCRATCH_TABLE, nj.OUTPUT_COLUMNS, "result_er_load_test.csv")
    counted = fetch_df(settings, f"SELECT COUNT(*) AS n FROM {SCRATCH_TABLE}")["n"].iloc[0]
    check("COPY loaded every row", loaded == counted == len(output), f"loaded {loaded}, in table {counted}")
    back = fetch_df(settings, f"SELECT * FROM {SCRATCH_TABLE} ORDER BY ts LIMIT 5")
finally:
    run_sql(f"DROP TABLE IF EXISTS {SCRATCH_TABLE}")
back

## Summary

In [ ]:
summary_df = pd.DataFrame(RESULTS, columns=["status", "test", "details"])
failed = summary_df[summary_df["status"] == "FAIL"]
print(f"{len(summary_df) - len(failed)} passed, {len(failed)} failed")
summary_df